# Tiingo historical stock data

See [Tiingo's website](https://www.tiingo.com/products/end-of-day-stock-price-data) for the historical data being studied in this notebook.

## Imports

In [1]:
import pandas as pd
import requests

## Initial request setup

### Token

In [2]:
token_path = '../../../../../.config/tiingo/token'
with open(token_path, 'r') as file:
    token = file.read().strip()

### Headers

In [3]:
headers = {
    'Content-Type': 'application/json',
    'Authorization': 'Token ' + token
}

## Microsoft data comparison

### MSFT data from yfinance

In [4]:
msft = pd.read_csv('../../data/yfinance/msft_daily.csv')
msft.iloc[-1]

Date            2026-03-03 00:00:00-05:00
Open                           393.140015
High                           406.700012
Low                            392.679993
Close                          403.929993
Volume                           37947881
Dividends                             0.0
Stock Splits                          0.0
Name: 10069, dtype: object

### MSFT data from Tiingo

In [5]:
url = 'https://api.tiingo.com/tiingo/daily/msft/prices?endDate=2026-03-03&format=csv&resampleFreq=daily'
# response = requests.get(url, headers=headers)

In [6]:
output = '../../data/tiingo/msft_daily.csv'
# with open(output, 'w') as file:
#     file.write(response.text)
# response

In [7]:
msft_tiingo = pd.read_csv(output)
msft_tiingo.iloc[-1]

date           2026-03-03
close              403.93
high                406.7
low                392.67
open               393.14
volume           38199209
adjClose           403.93
adjHigh             406.7
adjLow             392.67
adjOpen            393.14
adjVolume        38199209
divCash               0.0
splitFactor           1.0
Name: 10069, dtype: object

### Comparison

#### Total number of entries

In [8]:
# Note: need to select a column for apples-to-apples comparison
yf_size = msft['Open'].size
tiingo_size = msft_tiingo['adjOpen'].size
print("yf_size: ", yf_size)
print("tiingo_size: ", tiingo_size)

yf_size:  10070
tiingo_size:  10070


#### Dividend history

In [9]:
yf_dividend = msft['Dividends'].sum()
tiingo_dividend = msft_tiingo['divCash'].sum()
print("yfinance: ", yf_dividend)
print("tiingo: ", tiingo_dividend)

yfinance:  34.730000000000004
tiingo:  34.730000000000004


#### Stock splits

In [10]:
msft[msft['Stock Splits'] != 0]

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
385,1987-09-21 00:00:00-04:00,0.226433,0.242304,0.224317,0.226433,85550400,0.0,2.0
1034,1990-04-16 00:00:00-04:00,0.516352,0.522701,0.508945,0.514236,39470400,0.0,2.0
1338,1991-06-27 00:00:00-04:00,0.860234,0.866583,0.847537,0.863409,56510400,0.0,1.5
1582,1992-06-15 00:00:00-04:00,1.428433,1.485570,1.428433,1.442718,54940800,0.0,1.5
2072,1994-05-23 00:00:00-04:00,1.866486,1.942669,1.847440,1.926004,74947200,0.0,2.0
2717,1996-12-09 00:00:00-05:00,5.970852,6.237493,5.942283,6.227970,94718400,0.0,2.0
3020,1998-02-23 00:00:00-05:00,12.332146,12.446421,12.094074,12.436898,120803600,0.0,2.0
3296,1999-03-29 00:00:00-05:00,27.464012,28.225843,26.778364,28.149660,79777000,0.0,2.0
4273,2003-02-18 00:00:00-05:00,15.005029,15.230530,14.870946,15.212246,57415500,0.0,2.0


In [11]:
msft_tiingo[msft_tiingo['splitFactor'] != 1]

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor
385,1987-09-21,53.50,57.25,53.00,53.50,297044,0.226909,0.242814,0.224788,0.226909,42774336,0.0,2.0
1034,1990-04-16,60.75,61.75,60.13,61.00,274089,0.515316,0.523799,0.510057,0.517437,19734408,0.0,2.0
1338,1991-06-27,68.00,68.25,66.75,67.75,784844,0.865223,0.868404,0.849318,0.862042,37672512,0.0,1.5
1582,1992-06-15,75.75,78.00,75.00,75.00,1144600,1.445749,1.488692,1.431435,1.431435,36627200,0.0,1.5
2072,1994-05-23,50.56,51.00,48.50,49.00,2342100,1.929956,1.946751,1.851322,1.870408,37473600,0.0,2.0
2717,1996-12-09,81.75,81.87,78.00,78.37,5919900,6.241055,6.250216,5.954768,5.983015,47359200,0.0,2.0
3020,1998-02-23,81.62,81.69,79.37,80.94,15100450,12.462261,12.472949,12.118717,12.358434,60401800,0.0,2.0
3296,1999-03-29,92.37,92.62,87.87,90.12,19944250,28.207279,28.283622,26.833101,27.520190,39888500,0.0,2.0
4273,2003-02-18,24.96,24.99,24.40,24.62,28707750,15.244206,15.262529,14.902189,15.036553,28707750,0.0,2.0


#### Open pricing diffs
Note: Tiingo's bare values are way off.  I don't currently know the discrepancy, but a reasonable comparison requires using the adjusted values; e.g., "adjOpen" for the open on a particular day.

In [12]:
open_diffs = msft['Open'] - msft_tiingo['adjOpen']
open_diffs = open_diffs.to_frame(name='Open Diff')
open_diffs['Date'] = msft_tiingo['date']
open_diffs['Yahoo Open'] = msft['Open']
open_diffs['Tiingo Open'] = msft_tiingo['adjOpen']

print("Total absolute difference in columns: ", abs(open_diffs['Open Diff']).sum())

print("Difference greater than $0.10:")
open_diffs[abs(open_diffs['Open Diff']) > 0.1]

Total absolute difference in columns:  128.70388787649586
Difference greater than $0.10:


,Open Diff,Date,Yahoo Open,Tiingo Open
4631,0.184291,2004-07-21,18.378013,18.193721
5045,0.139656,2006-03-13,18.973054,18.833398
5051,-0.147267,2006-03-21,19.363962,19.511230
5618,0.426778,2008-06-20,20.830291,20.403513
5633,0.268033,2008-07-14,18.365250,18.097217
5745,-0.194066,2008-12-19,14.148406,14.342472
6193,0.214278,2010-10-01,18.738886,18.524607
6195,0.115877,2010-10-05,18.201764,18.085887
6216,0.101090,2010-11-03,20.773917,20.672827
6363,-0.121764,2011-06-06,18.411578,18.533342


#### High pricing diffs
I could see how open/close diffs may be related to timing or after-hours correction methodology.  These differences are to check if at least high/low data is consistent between the two datasets.

In [13]:
high_diffs = msft['High'] - msft_tiingo['adjHigh']
high_diffs = high_diffs.to_frame(name='High Diff')
high_diffs['Date'] = msft_tiingo['date']
high_diffs['Yahoo High'] = msft['High']
high_diffs['Tiingo High'] = msft_tiingo['adjHigh']

print("Total absolute difference in columns: ", abs(high_diffs['High Diff']).sum())

print("Difference greater than $0.10:")
high_diffs[abs(high_diffs['High Diff']) > 0.1]

Total absolute difference in columns:  126.0239427102048
Difference greater than $0.10:


,High Diff,Date,Yahoo High,Tiingo High
4943,-0.598271,2005-10-13,17.154243,17.752514
6219,1.379576,2010-11-08,21.840606,20.461030
7424,-0.536213,2015-08-24,37.045000,37.581213


#### Comparison summary
Tiingo seems to have the same dates as yfinance, but there are discrepancies in daily price data around the order of $0.01/day.  Eyeballing it, these errors do seem correlated over time (greater absolute price discrepancies for recent vs. older dates), but I suspect that's related to the _much_ larger value of the stock.  I.e., the relative error may be shrinking as the historical data approaches the present.

## Enron lookup

Enron's ticker was $ENE, so I want to see if Tiingo maintains pricing data from that time period.

In [16]:
url = 'https://api.tiingo.com/tiingo/daily/ene/prices?format=csv&resampleFreq=daily'
response = requests.get(url, headers=headers)
response.status_code

200

### Failure - not found

In [17]:
response.content

b"Error: Ticker 'ENE' not found"